# Part 1: Data Cleaning & Validation

Turn the dirty `ohlc_hw` table into a validated `ohlc_hw_cleaned` table, recording every decision.

**Order matters:** normalize tickers → remove duplicates → **diagnose** (before touching values)
→ repair within sorted ticker groups → quarantine what can't be repaired → validate → save.

The supplied `ohlc` table is used only as (a) each ticker's reference trading calendar and
(b) evidence: trusted date-specific adjustment factors and a check on suspicious prices.
It never replaces the dirty data wholesale. The raw `ohlc_hw` table is only read.

In [1]:
# Parameters
db_path = "data/data.db"  # local working copy, relative to this notebook (gitignored)
source_db = None  # None -> copy from Lectures/Lecture 4/data/data.db

In [2]:
import shutil
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

KEY = ["ticker", "ts"]
PRICES = ["open", "high", "low", "close"]  # raw prices: OHLC bounds apply to these only
VALUES = KEY + PRICES + ["adj_close", "volume"]
FLAGS = [f"{c}_repaired" for c in PRICES + ["adj_close", "volume"]]
REPORT_COLS = ["ticker", "issue_type", "issue_date", "description", "severity"]

In [3]:
# Work on a local copy so the course's source database is never modified.
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
db_path = Path(db_path)
if not db_path.is_file():
    source = Path(source_db) if source_db else course_root / "Lectures/Lecture 4/data/data.db"
    if not source.is_file():
        raise FileNotFoundError(f"Download data.db from bCourses and place it at {source}")
    db_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, db_path)
conn = sqlite3.connect(db_path)

## 1.1 Load and inspect

| column | meaning |
|---|---|
| `ts` | trading date |
| `ticker` | symbol as delivered (needs normalizing) |
| `open`, `high`, `low`, `close` | raw daily prices; must satisfy `low <= open, close <= high` |
| `adj_close` | close adjusted for splits/dividends (used for report returns); OHLC bounds don't apply |
| `volume` | shares traded |

In [4]:
raw = pd.read_sql_query("SELECT rowid AS source_row, * FROM ohlc_hw", conn)  # source_row traces each record
reference_raw = pd.read_sql_query("SELECT * FROM ohlc", conn)
raw.info()
display(raw.describe())
print(f"Date range: {raw['ts'].min()} to {raw['ts'].max()}")
print(f"Distinct raw ticker labels: {raw['ticker'].nunique()}; examples: {raw['ticker'].unique()[:3].tolist()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   source_row  397711 non-null  int64  
 1   ts          397711 non-null  object 
 2   open        397711 non-null  float64
 3   high        397711 non-null  float64
 4   low         397711 non-null  float64
 5   close       397711 non-null  float64
 6   adj_close   377856 non-null  float64
 7   volume      397711 non-null  int64  
 8   ticker      397711 non-null  object 
dtypes: float64(5), int64(2), object(2)
memory usage: 27.3+ MB


,source_row,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,198856.000000,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,114809.420792,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,1.000000,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,99428.500000,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,198856.000000,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,298283.500000,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,397711.000000,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


Date range: 1962-01-02 to 2024-12-06


Distinct raw ticker labels: 150; examples: ['CAT', 'CVX', 'HD']


### Normalize tickers before any grouping

Labels contain HTML markup, whitespace and mixed case. Grouping on raw strings would split one stock
into several, compute returns across interleaved fragments and hide duplicates.

In [5]:
def normalize(frame: pd.DataFrame) -> pd.DataFrame:
    """Strip HTML tags/whitespace and upper-case tickers (original kept as `raw_ticker`); parse `ts`."""
    out = frame.copy()
    out["raw_ticker"] = out["ticker"]
    out["ticker"] = (out["ticker"].astype("string")
                     .str.replace(r"<[^>]+>", "", regex=True).str.strip().str.upper())
    out["ts"] = pd.to_datetime(out["ts"], format="ISO8601").dt.normalize()
    return out


data_all = normalize(raw)
reference = normalize(reference_raw)
audit = data_all.groupby(["ticker", "raw_ticker"]).size().rename("rows").reset_index()
print(f"{data_all['raw_ticker'].nunique()} raw labels -> {data_all['ticker'].nunique()} tickers:")
print(sorted(data_all["ticker"].unique()))
display(audit.head(8))

150 raw labels -> 30 tickers:
['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


,ticker,raw_ticker,rows
0,AAPL,"<a target=""_blank""><span style=""color: blue"">A...",307
1,AAPL,"<a target=""_blank""><span style=""color: red"">AA...",318
2,AAPL,"<a target=""_blank""><span style=""font-weight: b...",306
3,AAPL,"<a target=""_blank""><span style=""font-weight: b...",264
4,AAPL,AAPL,10747
5,AMGN,"<a target=""_blank""><span style=""color: blue"">A...",294
6,AMGN,"<a target=""_blank""><span style=""color: red"">AM...",290
7,AMGN,"<a target=""_blank""><span style=""font-weight: b...",280


## 1.2 Detect data-quality issues

**Severity:** **high** = impossible/unusable, must be repaired or quarantined (missing or nonpositive
raw price, OHLC violation, invalid volume, adjusted close inconsistent with the trusted factor,
conflicting duplicate). **medium** = plausible but needs a documented decision (missing `adj_close`
or volume, missing trading day, 3σ return). **low** = harmless once handled or informational (exact
duplicate, zero volume).

In [6]:
def issues(data: pd.DataFrame, mask, issue_type: str, severity: str, description) -> pd.DataFrame:
    """Report rows (required schema) for the observations selected by `mask`."""
    rows = data.loc[mask, ["ticker", "ts"]].rename(columns={"ts": "issue_date"})
    text = description if isinstance(description, str) else description[mask]
    return rows.assign(issue_type=issue_type, severity=severity, description=text)[REPORT_COLS]


def reference_factors(reference: pd.DataFrame) -> pd.DataFrame:
    """Trusted date-specific adjustment factor (adj_close / close) and close from the `ohlc` table."""
    ref = reference.drop_duplicates(KEY)
    factor = ref["adj_close"] / ref["close"]
    return pd.DataFrame({"ticker": ref["ticker"], "ts": ref["ts"], "ref_close": ref["close"],
                         "ref_factor": factor.where(np.isfinite(factor) & factor.gt(0))})


def factor_mismatch(data: pd.DataFrame) -> pd.Series:
    """True where adj_close / close disagrees with the reference factor by more than 0.1%."""
    vendor = data["adj_close"] / data["close"]
    return vendor.notna() & data["ref_factor"].notna() & ~np.isclose(vendor, data["ref_factor"], rtol=1e-3)


def extreme_returns(data: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """3σ screen of simple close returns within each sorted ticker: |r - mean| > 3 * std (ddof=1).

    Both tails are flagged. Undefined returns, and tickers with zero/undefined std, are never flagged.
    This is a review flag, not a deletion rule. Returns (mask, returns).
    """
    close = data["close"].where(np.isfinite(data["close"]) & data["close"].gt(0))
    returns = close.groupby(data["ticker"]).pct_change(fill_method=None)
    grouped = returns.groupby(data["ticker"])
    mean, std = grouped.transform("mean"), grouped.transform("std")  # pandas std defaults to ddof=1
    return ((returns - mean).abs() > 3 * std) & std.gt(0), returns


def detect_issues(frame: pd.DataFrame, factors: pd.DataFrame) -> pd.DataFrame:
    """Data Quality Report for deduplicated rows; one observation may have several issues."""
    data = frame.sort_values(KEY).reset_index(drop=True).merge(factors, on=KEY, how="left")
    o, h, l, c, vol = (data[k] for k in PRICES + ["volume"])
    parts = [issues(data, data[col].isna(), f"missing_{col}",
                    "medium" if col in ("adj_close", "volume") else "high", f"{col} is missing")
             for col in PRICES + ["adj_close", "volume"]]
    for col in PRICES + ["adj_close"]:
        bad = np.isinf(data[col]) | data[col].le(0)
        parts.append(issues(data, bad, "invalid_price", "high", f"{col} = " + data[col].astype(str)))
    rule = pd.Series(np.select([h < l, h < c, l > c, (o < l) | (o > h)],
                               ["high < low", "high < close", "low > close", "open outside [low, high]"], ""),
                     index=data.index)
    parts.append(issues(data, rule.ne(""), "ohlc_bounds", "high", rule))
    parts.append(issues(data, factor_mismatch(data), "adj_factor_mismatch", "high",
                        "adj_close/close " + (data["adj_close"] / c).round(4).astype(str)
                        + " vs reference " + data["ref_factor"].round(4).astype(str)))
    parts.append(issues(data, np.isinf(vol) | vol.lt(0), "invalid_volume", "high", "volume = " + vol.astype(str)))
    parts.append(issues(data, vol.eq(0), "zero_volume", "low", "zero volume: possibly stale data"))
    extreme, returns = extreme_returns(data)
    parts.append(issues(data, extreme, "extreme_return", "medium",
                        "return " + returns.map("{:+.2%}".format) + " beyond 3 std of ticker mean"))
    return pd.concat([p for p in parts if len(p)] or [pd.DataFrame(columns=REPORT_COLS)], ignore_index=True)


def missing_sessions(frame: pd.DataFrame, expected_dates) -> pd.DatetimeIndex:
    """Expected dates absent from one ticker's observations."""
    return pd.DatetimeIndex(expected_dates).difference(pd.DatetimeIndex(frame["ts"])).sort_values()


def gap_report(data: pd.DataFrame, reference: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Compare each ticker with its own dates in `ohlc`, so exchange holidays never count as gaps."""
    rows, summary = [], []
    for ticker, ref in reference.groupby("ticker"):
        gaps = missing_sessions(data[data["ticker"] == ticker], ref["ts"].unique())
        summary.append(f"Ticker {ticker} is missing {len(gaps)} trading days between "
                       f"{ref['ts'].min().date()} and {ref['ts'].max().date()}")
        rows += [{"ticker": ticker, "issue_type": "missing_session", "issue_date": d,
                  "description": "in reference calendar but absent", "severity": "medium"} for d in gaps]
    return pd.DataFrame(rows, columns=REPORT_COLS), summary

### Duplicates

Exact duplicates (identical after normalization) are dropped. A `(ticker, ts)` key with *different*
values has no defensible "first" record, so after displaying them all its rows are quarantined.

In [7]:
def deduplicate(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return (unique rows, removed exact duplicates, conflicting-key rows for quarantine)."""
    exact = data.duplicated(VALUES)
    removed = data[exact].assign(reason="exact duplicate")
    data = data[~exact]
    conflict = data.duplicated(KEY, keep=False)
    return data[~conflict].copy(), removed, data[conflict].assign(reason="conflicting duplicate key")

## 1.3 Clean

**Decisions**

* **All four raw prices missing** → row dropped *before* filling.
* **Missing or nonpositive raw price, or open/close outside [low, high]** → set to NaN, then
  forward-filled within the ticker (**Option 1**). If the carried value would break *today's* range,
  today's (high + low) / 2 is used (**Option 3**); high/low are then widened to contain open/close.
  For an impossible relationship the open/close is treated as the bad field, since it disagrees with two
  other prices that agree with each other. `high < low` can't tell us which bound is wrong → quarantine.
* **3σ returns** are classified with evidence before anything changes: raw close >1% from the reference
  → `data_error` (interpolate the row between valid neighbours, interior only); reference factor changes
  that day → `corporate_action` (keep); matches the reference → `confirmed_by_reference` (keep; we don't
  claim a news cause we can't see); no reference → `unverified` (keep). The flag is recomputed afterwards.
* **Unknown volume** → 0 with `volume_repaired` set: unknown volume is not evidence of no trading.
* **Adjusted close** → wherever missing, invalid, inconsistent with the trusted factor, or the raw close
  was repaired: `validated close × reference factor` for the same date. Factors are never forward-filled.
* **Errors shared with the reference.** `ohlc_hw`'s raw prices are a copy of `ohlc`, so the reference is
  not independent evidence: an error present in both looks "confirmed". The clearest such error is a
  one-day round trip in the *adjusted* series (|log return| > 25% that the next day reverses to within 5%),
  which no real price path produces. Those rows (`reference_error`) are interpolated between neighbours
  and `adj_close` is rebuilt with the neighbouring days' factor, if they agree within 1% (no split);
  otherwise the row is quarantined.
* **Leading gaps** (nothing earlier to carry) and rows without a trusted factor are quarantined, never
  extrapolated. Every changed cell sets `<column>_repaired`, and raw OHLC bounds and adjusted-price
  consistency are rechecked after each repair step.

In [8]:
SPIKE, SPIKE_REVERSAL = 0.25, 0.05  # one-day round trip: |log r| > 25%, next day returns to within 5%


def assert_bounds(data: pd.DataFrame, step: str) -> None:
    """Recheck low <= open, close <= high on complete rows after a repair step.

    Rows with high < low are excluded here; they are quarantined at the end.
    """
    complete = data[PRICES].notna().all(axis=1) & data["high"].ge(data["low"])
    inside = data["open"].between(data["low"], data["high"]) & data["close"].between(data["low"], data["high"])
    assert not (complete & ~inside).any(), f"{(complete & ~inside).sum()} rows break OHLC bounds after {step}"


def classify_extremes(data: pd.DataFrame) -> pd.Series:
    """Evidence-based class for each 3σ return flag ('' where not flagged)."""
    extreme, _ = extreme_returns(data)
    prev_factor = data.groupby("ticker")["ref_factor"].shift()
    has_ref = data["ref_close"].notna()
    conditions = [~extreme, data["close_repaired"],
                  has_ref & ~np.isclose(data["close"], data["ref_close"], rtol=0.01),
                  prev_factor.notna() & ~np.isclose(data["ref_factor"], prev_factor, rtol=1e-3),
                  has_ref]
    classes = ["", "repaired_value", "data_error", "corporate_action", "confirmed_by_reference"]
    return pd.Series(np.select(conditions, classes, "unverified"), index=data.index)


def interpolate_rows(data: pd.DataFrame, rows: pd.Series) -> None:
    """Replace raw OHLC on `rows` by linear interpolation between valid neighbours (in place, interior only).

    Interpolating each price between two valid rows keeps low <= open, close <= high.
    """
    for col in PRICES:
        filled = data[col].mask(rows).groupby(data["ticker"]).transform(lambda s: s.interpolate(limit_area="inside"))
        data.loc[rows, col] = filled[rows]
        data.loc[rows, f"{col}_repaired"] = True


def repair(frame: pd.DataFrame, factors: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Repair raw prices, extreme-return errors, volume and adjusted close within sorted tickers.

    Returns (repaired rows, quarantined rows with a `reason`).
    """
    data = frame.sort_values(KEY).reset_index(drop=True).merge(factors, on=KEY, how="left")

    # 1. Invalid prices, and open/close outside a consistent [low, high], become missing.
    for col in PRICES:
        bad = data[col].isna() | np.isinf(data[col]) | data[col].le(0)
        if col in ("open", "close"):
            bad |= data["high"].ge(data["low"]) & ~data[col].between(data["low"], data["high"])
        data[col] = data[col].mask(bad)
        data[f"{col}_repaired"] = bad

    # 2. Forward-fill within the ticker; a carried open/close outside today's range uses the midpoint.
    for col in ["high", "low"]:
        data[col] = data.groupby("ticker")[col].ffill()
    midpoint = (data["high"] + data["low"]) / 2
    for col in ["open", "close"]:
        carried = data.groupby("ticker")[col].ffill()
        candidate = carried.where(carried.between(data["low"], data["high"]), midpoint.where(carried.notna()))
        data[col] = data[col].fillna(candidate)  # only cells set to NaN in step 1 change
    touched = data[FLAGS[:4]].any(axis=1)
    for col, how in (("high", "max"), ("low", "min")):
        bound = data[["open", col, "close"]].agg(how, axis=1)
        widen = touched & bound.notna() & data[col].ne(bound)
        data.loc[widen, col] = bound[widen]
        data.loc[widen, f"{col}_repaired"] = True
    assert_bounds(data, "forward-fill")

    # 3. Classify 3σ returns, then interpolate confirmed data errors (inside gaps only).
    data["extreme_return_class"] = classify_extremes(data)
    error = data["extreme_return_class"].eq("data_error")
    interpolate_rows(data, error)
    assert_bounds(data, "data-error interpolation")

    # 4. Unknown volume -> 0 placeholder, flagged.
    unknown = data["volume"].isna() | np.isinf(data["volume"]) | data["volume"].lt(0)
    data["volume"] = data["volume"].mask(unknown, 0)
    data["volume_repaired"] = unknown

    # 5. Adjusted close from validated close x trusted factor (NaN if no factor -> quarantined below).
    adj = data["adj_close"]
    rebuild = adj.isna() | np.isinf(adj) | adj.le(0) | factor_mismatch(data) | data["close_repaired"]
    data.loc[rebuild, "adj_close"] = (data["close"] * data["ref_factor"])[rebuild]
    data["adj_close_repaired"] = rebuild
    assert not factor_mismatch(data).any(), "adj_close inconsistent with the reference after rebuild"

    # 6. One-day round trips in the adjusted series are errors shared with the reference: interpolate the
    #    row and rebuild adj_close with the neighbouring days' factor when they agree, else quarantine.
    by_ticker = lambda s: s.groupby(data["ticker"])
    r = np.log(data["adj_close"] / by_ticker(data["adj_close"]).shift())
    spike = r.abs().gt(SPIKE) & (r + by_ticker(r).shift(-1)).abs().lt(SPIKE_REVERSAL)
    factor = data["adj_close"] / data["close"]
    prev_factor, next_factor = by_ticker(factor).shift(), by_ticker(factor).shift(-1)
    fixable = spike & np.isclose(prev_factor, next_factor, rtol=0.01)  # no split between; dividend drift ok
    interpolate_rows(data, fixable)
    data.loc[fixable, "adj_close"] = (data["close"] * prev_factor)[fixable]
    data.loc[spike & ~fixable, "adj_close"] = np.nan  # no consistent factor -> quarantined below
    data.loc[spike, "adj_close_repaired"] = True
    data.loc[spike, "extreme_return_class"] = "reference_error"
    assert_bounds(data, "reference-error interpolation")
    data["extreme_return_flag"] = extreme_returns(data)[0]  # review flag on the fully repaired series

    # 7. Quarantine whatever is still unusable.
    reason = pd.Series(np.select(
        [data[PRICES + ["adj_close"]].isna().any(axis=1), data["high"] < data["low"]],
        ["unresolved price: nothing earlier to carry forward / no trusted factor",
         "high < low: cannot tell which bound is wrong"], ""), index=data.index)
    bad = reason.ne("")
    data = data.drop(columns=["ref_close", "ref_factor"])
    return data[~bad].reset_index(drop=True), data[bad].assign(reason=reason[bad])


def validate(clean: pd.DataFrame, factors: pd.DataFrame) -> pd.Series:
    """Pass/fail for every rule `ohlc_hw_cleaned` must satisfy."""
    prices = clean[PRICES + ["adj_close"]]
    merged = clean.merge(factors, on=KEY, how="left")
    # reference_error rows deliberately use the neighbours' factor instead of the (wrong) reference one.
    inconsistent = factor_mismatch(merged) & merged["extreme_return_class"].ne("reference_error")
    return pd.Series({
        "prices present, finite and positive": (np.isfinite(prices) & prices.gt(0)).all(axis=None),
        "unique (ticker, ts) keys": not clean.duplicated(KEY).any(),
        "volume finite and nonnegative": (np.isfinite(clean["volume"]) & clean["volume"].ge(0)).all(),
        "low <= open <= high": clean["open"].between(clean["low"], clean["high"]).all(),
        "low <= close <= high": clean["close"].between(clean["low"], clean["high"]).all(),
        "adj_close matches trusted factor": not inconsistent.any(),
    }, name="passed")


def run_pipeline(raw_frame: pd.DataFrame, reference: pd.DataFrame) -> dict:
    """Normalize, deduplicate, diagnose (before repairs), repair and quarantine one delivery."""
    data, removed, conflicts = deduplicate(normalize(raw_frame))
    factors = reference_factors(reference)
    gaps, gap_summary = gap_report(pd.concat([data, conflicts]), reference)
    parts = [issues(removed, slice(None), "exact_duplicate", "low", "identical record; removed"),
             issues(conflicts, slice(None), "conflicting_duplicate", "high", "same key, different values"),
             detect_issues(data, factors), gaps]
    report = pd.concat([p for p in parts if len(p)], ignore_index=True)  # skip detectors that found nothing
    all_missing = data[PRICES].isna().all(axis=1)
    removed = pd.concat([removed, data[all_missing].assign(reason="all raw prices missing")])
    clean, quarantined = repair(data[~all_missing], factors)
    return {"report": report.sort_values(["ticker", "issue_date"], ignore_index=True), "gap_summary": gap_summary,
            "conflicts": conflicts, "clean": clean, "removed": removed,
            "quarantined": pd.concat([conflicts, quarantined]), "factors": factors}

## Injected examples

The real table may not contain every required fault (finding zero is a valid result), so a tiny
delivery with one of each is run through the *same* pipeline.

In [9]:
rng = np.random.default_rng(7)
dates = pd.bdate_range("2024-01-01", periods=30)
day = lambda i: dates[i]
split = np.where(np.arange(30) < 15, 0.5, 1.0)  # SPLT: 2:1 split on day 15
close = np.r_[100 * np.exp(np.cumsum(rng.normal(0, 0.01, 30))), 100 * np.exp(np.cumsum(rng.normal(0, 0.01, 30))) / split]
demo_ref = pd.DataFrame({"ts": np.tile(dates.strftime("%Y-%m-%d"), 2), "ticker": ["DEMO"] * 30 + ["SPLT"] * 30,
                         "open": close, "high": close * 1.005, "low": close * 0.995, "close": close,
                         "adj_close": close * np.r_[np.ones(30), split], "volume": 1000.0})
demo_ref.loc[27, PRICES + ["adj_close"]] *= 2  # one-day spike in the REFERENCE itself (copied into raw)
demo_raw = demo_ref.drop(index=3).assign(source_row=lambda f: f.index)  # day 3 absent -> missing session
demo_raw.loc[0, "ticker"] = " <b>demo</b> "    # messy label
demo_raw.loc[1, PRICES] = np.nan               # all prices missing -> dropped
demo_raw.loc[5, "close"] *= 1.05               # close above high
demo_raw.loc[7, "open"] = -1.0                 # nonpositive price
demo_raw.loc[9, "volume"] = np.nan             # unknown volume
demo_raw.loc[14, PRICES] *= 10                 # consistent 10x row: only the return screen sees it
demo_raw.loc[17, "adj_close"] = np.nan         # missing adj_close
demo_raw.loc[19, "adj_close"] *= 12            # inflated adj_close
demo_raw.loc[20, "open"] = np.nan              # one missing price
demo_raw.loc[30, "close"] = np.nan             # SPLT leading gap
demo_raw = pd.concat([demo_raw, demo_raw.loc[[22]],                                  # exact duplicate
                      demo_raw.loc[[24]].assign(close=lambda f: f["close"] * 1.001)],  # conflicting duplicate
                     ignore_index=True)

demo = run_pipeline(demo_raw, normalize(demo_ref))
display(demo["report"])
print("\n".join(demo["gap_summary"]))
display(demo["quarantined"][["ticker", "ts", "reason"]])

,ticker,issue_type,issue_date,description,severity
0,DEMO,missing_open,2024-01-02,open is missing,high
1,DEMO,missing_high,2024-01-02,high is missing,high
2,DEMO,missing_low,2024-01-02,low is missing,high
3,DEMO,missing_close,2024-01-02,close is missing,high
4,DEMO,missing_session,2024-01-04,in reference calendar but absent,medium
5,DEMO,ohlc_bounds,2024-01-08,high < close,high
6,DEMO,adj_factor_mismatch,2024-01-08,adj_close/close 0.9524 vs reference 1.0,high
7,DEMO,invalid_price,2024-01-10,open = -1.0,high
8,DEMO,ohlc_bounds,2024-01-10,"open outside [low, high]",high
9,DEMO,missing_volume,2024-01-12,volume is missing,medium


Ticker DEMO is missing 1 trading days between 2024-01-01 and 2024-02-09
Ticker SPLT is missing 0 trading days between 2024-01-01 and 2024-02-09


,ticker,ts,reason
23,DEMO,2024-02-02,conflicting duplicate key
60,DEMO,2024-02-02,conflicting duplicate key
27,SPLT,2024-01-01,unresolved price: nothing earlier to carry for...


In [10]:
r, c = demo["report"], demo["clean"].set_index(KEY)
row = lambda i, t="DEMO": c.loc[(t, day(i))]
assert r.query("issue_type == 'missing_session'")["issue_date"].tolist() == [day(3)]
assert {"missing_open", "ohlc_bounds", "invalid_price", "missing_volume", "extreme_return", "missing_adj_close",
        "adj_factor_mismatch", "exact_duplicate", "conflicting_duplicate"} <= set(r["issue_type"])
assert all(("DEMO", day(i)) not in c.index for i in (1, 3, 24))   # dropped / absent / conflict: nothing invented
assert ("SPLT", day(0)) not in c.index                             # leading gap quarantined
assert row(5)["close_repaired"] and row(5)["low"] <= row(5)["close"] <= row(5)["high"]
assert row(7)["open_repaired"] and row(20)["open_repaired"]
assert row(9)["volume_repaired"] and row(9)["volume"] == 0
assert row(14)["extreme_return_class"] == "data_error"
assert np.isclose(row(14)["close"], (row(13)["close"] + row(15)["close"]) / 2)
assert row(15, "SPLT")["extreme_return_class"] == "corporate_action"
assert row(27)["extreme_return_class"] == "reference_error" and row(27)["adj_close_repaired"]
assert np.isclose(row(27)["close"], (row(26)["close"] + row(28)["close"]) / 2)  # spike removed, not "confirmed"
assert all(row(i)["adj_close_repaired"] and np.isclose(row(i)["adj_close"], demo_ref.loc[i, "adj_close"]) for i in (17, 19))
assert validate(demo["clean"], demo["factors"]).all()
print("All injected-example checks passed.")

All injected-example checks passed.


## Run on `ohlc_hw`

In [11]:
result = run_pipeline(raw, reference)
report, clean = result["report"], result["clean"]
print(f"Data Quality Report (before repairs): {len(report):,} rows")
display(report.groupby(["issue_type", "severity"]).size().rename("issues").reset_index())
display(report[report["issue_type"] != "exact_duplicate"].head(10))
print("Conflicting keys found:", len(result["conflicts"]))
print("\n".join(result["gap_summary"]))

Data Quality Report (before repairs): 63,392 rows


,issue_type,severity,issues
0,adj_factor_mismatch,high,10417
1,exact_duplicate,low,29460
2,extreme_return,medium,5056
3,invalid_price,high,5
4,missing_adj_close,medium,18374
5,ohlc_bounds,high,5
6,zero_volume,low,75


,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26,return +9.23% beyond 3 std of ticker mean,medium
3,AAPL,missing_adj_close,1981-01-08,adj_close is missing,medium
4,AAPL,missing_adj_close,1981-01-13,adj_close is missing,medium
5,AAPL,adj_factor_mismatch,1981-02-03,adj_close/close 10.2537 vs reference 0.7709,high
7,AAPL,missing_adj_close,1981-02-12,adj_close is missing,medium
12,AAPL,extreme_return,1981-04-02,return +8.77% beyond 3 std of ticker mean,medium
16,AAPL,missing_adj_close,1981-04-16,adj_close is missing,medium
18,AAPL,adj_factor_mismatch,1981-04-28,adj_close/close 16.1485 vs reference 0.7709,high
19,AAPL,adj_factor_mismatch,1981-05-08,adj_close/close 17.0343 vs reference 0.7709,high
20,AAPL,missing_adj_close,1981-05-19,adj_close is missing,medium


Conflicting keys found: 0
Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-

In [12]:
quarantined = result["quarantined"]
print("Extreme-return evidence (before interpolation):")
display(clean["extreme_return_class"].value_counts().drop("", errors="ignore"))
display(clean[clean["extreme_return_class"].ne("")].groupby("extreme_return_class").head(2)[KEY + ["close", "extreme_return_class"]])
print("One-day round trips found in the adjusted series (errors shared with the reference):")
display(clean[clean["extreme_return_class"].eq("reference_error")][KEY + PRICES + ["adj_close"]])
display(quarantined[quarantined["extreme_return_class"].eq("reference_error")][KEY + ["reason"]]
        if "extreme_return_class" in quarantined else None)

Extreme-return evidence (before interpolation):


extreme_return_class
confirmed_by_reference    4233
corporate_action           820
reference_error              3
Name: count, dtype: int64

,ticker,ts,close,extreme_return_class
9,AAPL,1980-12-26,0.15848,confirmed_by_reference
76,AAPL,1981-04-02,0.11775,confirmed_by_reference
2486,AAPL,1990-10-19,0.28000,corporate_action
3321,AAPL,1994-02-07,0.32589,corporate_action
318327,TRV,1976-10-28,4.65000,reference_error
318868,TRV,1978-12-20,4.05500,reference_error


One-day round trips found in the adjusted series (errors shared with the reference):


,ticker,ts,open,high,low,close,adj_close
318327,TRV,1976-10-28,4.650,4.650,4.650,4.650,0.988967
318868,TRV,1978-12-20,4.055,4.055,4.055,4.055,0.914342
319533,TRV,1981-08-10,6.030,6.030,6.030,6.030,1.570000


,ticker,ts,reason


### Row accounting, repairs and validation

In [13]:
removed, quarantined = result["removed"], result["quarantined"]
print(f"raw {len(raw):,} = retained {len(clean):,} + removed {len(removed):,} + quarantined {len(quarantined):,}")
assert len(clean) + len(removed) + len(quarantined) == len(raw)
display(pd.concat([removed, quarantined])["reason"].value_counts())
display(clean[FLAGS + ["extreme_return_flag"]].sum().rename("rows"))
checks = validate(clean, result["factors"])
display(checks.to_frame())
assert checks.all(), "cleaned data failed validation"

raw 397,711 = retained 368,246 + removed 29,460 + quarantined 5


reason
exact duplicate                                                           29460
unresolved price: nothing earlier to carry forward / no trusted factor        5
Name: count, dtype: int64

open_repaired              3
high_repaired              3
low_repaired               3
close_repaired             3
adj_close_repaired     28793
volume_repaired            0
extreme_return_flag     5175
Name: rows, dtype: int64

,passed
"prices present, finite and positive",True
"unique (ticker, ts) keys",True
volume finite and nonnegative,True
low <= open <= high,True
low <= close <= high,True
adj_close matches trusted factor,True


In [14]:
# Rerun the diagnosis on the cleaned data: only review flags (3σ returns, zero volume) should remain,
# plus gaps where a removed/quarantined row left a date absent.
after = detect_issues(clean, result["factors"])
# reference_error rows deliberately differ from the (wrong) reference factor, so that check is expected there.
deliberate = set(map(tuple, clean.loc[clean["extreme_return_class"].eq("reference_error"), KEY].to_numpy()))
expected = after["issue_type"].eq("adj_factor_mismatch") & [k in deliberate for k in zip(after["ticker"], after["issue_date"])]
after = after[~expected]
gaps_after, _ = gap_report(clean, reference)
display(pd.concat([after, gaps_after]).groupby(["issue_type", "severity"]).size().rename("issues").reset_index())
assert after["severity"].ne("high").all()

C:\Users\kantr\AppData\Local\Temp\ipykernel_32360\3570542359.py:6: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  expected = after["issue_type"].eq("adj_factor_mismatch") & [k in deliberate for k in zip(after["ticker"], after["issue_date"])]


,issue_type,severity,issues
0,extreme_return,medium,5176
1,missing_session,medium,5
2,zero_volume,low,74


In [15]:
cols = PRICES + ["adj_close", "volume"]
before = data_all.drop_duplicates(VALUES)
display(pd.concat({"before": before[cols].describe().T, "after": clean[cols].describe().T}, axis=1)
        .loc[:, (slice(None), ["count", "mean", "min", "max"])].round(2))
display(pd.DataFrame({"missing before": before[cols].isna().sum(), "missing after": clean[cols].isna().sum()}))
coverage = clean.groupby("ticker")["ts"].agg(first="min", last="max", days="count")
print("Histories end on different dates; reports must disclose actual coverage:")
display(coverage["last"].value_counts())

before                                      after               \
              count         mean   min           max     count         mean   
open       368251.0        46.31  0.00  6.042600e+02  368246.0        46.31   
high       368251.0        46.77  0.01  6.086300e+02  368246.0        46.77   
low        368251.0        45.84  0.00  5.978800e+02  368246.0        45.84   
close      368251.0        46.32  0.01  6.054000e+02  368246.0        46.32   
adj_close  349877.0        51.71  0.00  1.044251e+04  368246.0        37.27   
volume     368251.0  30951422.27  0.00  9.276606e+09  368246.0  30951838.94   

                               
            min           max  
open       0.01  6.042600e+02  
high       0.01  6.086300e+02  
low        0.00  5.978800e+02  
close      0.01  6.054000e+02  
adj_close  0.00  6.054000e+02  
volume     0.00  9.276606e+09

,missing before,missing after
open,0,0
high,0,0
low,0,0
close,0,0
adj_close,18374,0
volume,0,0


Histories end on different dates; reports must disclose actual coverage:


last
2024-10-18    22
2024-12-06     8
Name: count, dtype: int64

## Save `ohlc_hw_cleaned`

`ts` is stored as ISO `YYYY-MM-DD` text so SQL date filters work; repair flags, the original label and
`source_row` are saved with the data. The raw table is untouched.

In [16]:
out = clean[VALUES + ["raw_ticker", "source_row"] + FLAGS + ["extreme_return_flag", "extreme_return_class"]].copy()
out["ts"] = out["ts"].dt.strftime("%Y-%m-%d")
with conn:
    out.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
display(pd.read_sql_query("SELECT COUNT(*) AS rows, MIN(ts) AS first, MAX(ts) AS last FROM ohlc_hw_cleaned "
                          "WHERE ts BETWEEN '2024-06-01' AND '2024-11-30'", conn))
assert conn.execute("SELECT COUNT(*) FROM ohlc_hw").fetchone()[0] == len(raw), "raw table changed"
conn.close()

,rows,first,last
0,3142,2024-06-03,2024-11-29
